# <font size=6><b>Lec08.Doc2Vec으로 공시 사업보고서 유사도
* 

Doc2Vec : w2v는 주변 단어 보고 중간 단어 맞추는 느낌이면 d2v는 문서 번호 개념 추가해서 문서전체를 하나의 좌표로 찍는 기술<br>
**Doc2Vec은 문서의 '길이'에 상관없이 고정된 크기의 벡터를 만든다**<br>
따라서 서로 길이가 다른 보고서끼리도 1:1로 유사도를 비교할 수 있다

In [3]:
#!gdown https://drive.google.com/uc?id=1XS0UlE8gNNTRjnL6e64sMacOhtVERIqL

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from konlpy.tag import Mecab
from gensim.models.doc2vec import TaggedDocument
from gensim.models import doc2vec

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# data load

In [6]:
df = pd.read_csv("./dataset/dart.csv")
df.head(3)

,code,market,name,business
0,000020,KOSPI,동화약품,II. 사업의 내용\n1. 사업의 개요\n가. 일반적인 사항\n기업회계기준서 제11...
1,000040,KOSPI,KR모터스,II. 사업의 내용\n1. 사업의 개요\n가. 업계의 현황\n수출주력시장인 유럽 불...
2,000050,KOSPI,경방,II. 사업의 내용\n1. 사업의 개요\n(1) 산업의 특성\n[섬유사업부문]\n면...


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2589 entries, 0 to 2588
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   code      2589 non-null   object
 1   market    2589 non-null   object
 2   name      2589 non-null   object
 3   business  2295 non-null   object
dtypes: object(4)
memory usage: 81.0+ KB


In [9]:
print(df.shape)
df = df.dropna(subset= "business")
print(df.shape)

(2589, 4)
(2295, 4)


# 토큰화

In [10]:
from konlpy.tag import Mecab
mecab = Mecab(dicpath=r"C:/mecab/mecab-ko-dic")  #------- 역슬러시 인식 불가 , 슬러시 사용

tagged_corpus_list = []
for index, row in tqdm(df.iterrows(), total=len(df)):
  text = row['business']
  tag = row['name']
  tagged_corpus_list.append(TaggedDocument(tags=[tag], words=mecab.morphs(text)))

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2295/2295 [00:46<00:00, 49.18it/s]


In [13]:
print('문서의 수 :', len(tagged_corpus_list))
#tagged_corpus_list[0]

문서의 수 : 2295


In [15]:
#핵심부분 

model = doc2vec.Doc2Vec(vector_size=300, alpha=0.025, min_alpha=0.025, window=8)

# Vocabulary 빌드
model.build_vocab(tagged_corpus_list)

# Doc2Vec 학습
model.train(tagged_corpus_list, total_examples=model.corpus_count, epochs=20)

# 모델 저장
model.save('dart.doc2vec')

In [17]:
similar_doc = model.dv.most_similar('동화약품')
print(similar_doc)

[('종근당', 0.5607609748840332), ('제일약품', 0.5362198352813721), ('일양약품', 0.5153180360794067), ('동아에스티', 0.5126904249191284), ('삼아제약', 0.5009893178939819), ('경동제약', 0.4969020485877991), ('대웅제약', 0.4959961473941803), ('조아제약', 0.48876678943634033), ('동국제약', 0.48818710446357727), ('한미약품', 0.4809749722480774)]
